# candidate_colab.ipynb

Builds scikit-image v0.18.3 from scratch, applies our vectorized
fast-path patch to `regionprops_table`, and runs the same
workload as `baseline_colab.ipynb` so `tests.ipynb` can compare
outputs.

## What the patch does

The original `regionprops_table` walks every region and accesses
every requested property through `_RegionProperties.__getitem__`,
paying full Python-attribute-lookup + cache-check overhead per
(region, property) pair. For workloads with thousands of small
regions and a handful of common scalar properties (area, bbox,
centroid, mean / min / max intensity, ...), that dispatch is the
wall-clock cost — the actual math is trivial.

Our patch adds a vectorized fast path:

- `scipy.ndimage.find_objects` gives every bounding box in one C call.
- `np.bincount` gives areas and centroid sums for ALL labels at
  once.
- `scipy.ndimage.minimum` / `.maximum` give per-label intensity
  extrema in one C pass each.
- `regionprops_table` dispatches to this fast path **only** when
  all requested properties are in the supported set and
  `extra_properties is None`. Otherwise the original
  implementation runs unchanged — there is no behaviour change
  for any other call.

Trade-offs (full discussion in `README.md`):
- The fast path scans the **whole image** (incl. background)
  rather than only foreground pixels, so it has a small
  per-pixel overhead that doesn't help when n_regions is tiny.
  For typical many-region workloads this is dwarfed by the
  Python savings.
- Slight extra memory: ~5x H·W·8 bytes of float coord
  buffers + bincount outputs (peak ~250 MB on a 4096x4096 image).
  Inside Colab CPU's 12.7 GB RAM this is comfortable.
- Floating-point results may differ from the baseline at the
  1e-12 level because `np.bincount` accumulates in a different
  order than per-region `np.sum`. Tolerance documented and
  checked in `tests.ipynb` (1e-6 relative).


## 1. Runtime info

In [ ]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version


## 1.5 Config — clone this submission repo

In [ ]:
# ====================================================================
# CONFIG (only thing the evaluator may need to change)
# --------------------------------------------------------------------
# Set the URL of THIS submission's GitHub repo (the one containing
# baseline_colab.ipynb, candidate_colab.ipynb, tests.ipynb, etc.).
# If you set the env var SUBMISSION_REPO_URL before opening Colab,
# this falls through to that value.
# ====================================================================
import os
SUBMISSION_REPO_URL = os.environ.get(
    "SUBMISSION_REPO_URL",
    "https://github.com/OWNER/REPO.git"   # <-- EDIT THIS if not set via env
)
SUBMISSION_DIR = "/content/submission"

if "OWNER/REPO" in SUBMISSION_REPO_URL:
    print("=" * 70)
    print("WARNING: SUBMISSION_REPO_URL is the placeholder.")
    print("Edit this cell to point at your fork, OR set:")
    print("  os.environ['SUBMISSION_REPO_URL'] = "
          "'https://github.com/yourname/yourrepo.git'")
    print("before re-running this cell.")
    print("=" * 70)
    raise SystemExit(0)

if not os.path.isdir(SUBMISSION_DIR):
    !git clone --depth 1 $SUBMISSION_REPO_URL $SUBMISSION_DIR

import sys
for p in (SUBMISSION_DIR,
          os.path.join(SUBMISSION_DIR, "workloads"),
          os.path.join(SUBMISSION_DIR, "benchmark"),
          os.path.join(SUBMISSION_DIR, "patches")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("Submission repo at:", SUBMISSION_DIR)


## 2. Clone scikit-image + install

In [ ]:
import os
WORK = "/content/work"
CANDIDATE_DIR = os.path.join(WORK, "scikit-image-candidate")
os.makedirs(WORK, exist_ok=True)
if not os.path.isdir(CANDIDATE_DIR):
    !git clone --depth 1 --branch v0.18.3 https://github.com/scikit-image/scikit-image.git {CANDIDATE_DIR}


In [ ]:
# ---- Environment pinning (avoids dependency drift in Colab) -------------
# scikit-image v0.18.3 needs numpy < 1.24 and a compatible scipy.
!pip install -q --upgrade pip setuptools wheel
!pip install -q "numpy==1.23.5" "scipy==1.10.1" "cython<3" pytest line_profiler
!pip install -q networkx imageio tifffile pillow PyWavelets

import importlib, sys
for mod in ("numpy", "scipy"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
import numpy as np, scipy
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)


In [ ]:
%cd $CANDIDATE_DIR
!pip install -q -e . 2>&1 | tail -10
%cd /content
import importlib, sys
for k in [k for k in sys.modules if k.startswith('skimage')]:
    del sys.modules[k]
import skimage
print("skimage version:", skimage.__version__)
assert skimage.__version__ == "0.18.3"


## 3. Apply the optimization patch

In [ ]:
!python $SUBMISSION_DIR/patches/apply_optimization.py


In [ ]:
# Verify the patch is active.
import importlib, sys
for k in [k for k in sys.modules if k.startswith('skimage')]:
    del sys.modules[k]
from skimage.measure import regionprops_table
from skimage.measure import _regionprops_fast  # raises if patch failed
print("fast path module:", _regionprops_fast.__file__)
!python $SUBMISSION_DIR/patches/apply_optimization.py --check


## 4. Build the workload (same as baseline)

In [ ]:
from generate_workload import build_workload, WORKLOAD_PROPERTIES
import time, numpy as np
t0 = time.perf_counter()
labels, intensity, n_regions = build_workload()
print(f"workload built in {time.perf_counter()-t0:.2f}s | "
      f"shape={labels.shape}, n_regions={n_regions}")


## 5. Run the candidate workload + save output

We deliberately do NOT compute speedup or correctness here — that
belongs in `tests.ipynb` so concerns stay separated.


In [ ]:
os.makedirs("/content/outputs", exist_ok=True)
out = regionprops_table(labels, intensity_image=intensity,
                        properties=list(WORKLOAD_PROPERTIES))
np.savez_compressed("/content/outputs/candidate_output.npz", **out)
print("candidate_output.npz columns:", sorted(out.keys()))
print("first 3 rows of 'area':", out["area"][:3])
